# 不计分实验：Logistic Regression 的 Gradient Descent

在本实验中，你将实现 logistic regression 的 gradient 计算，用于 gradient descent 的参数更新。

## 数据集

我们继续使用之前的数据集。


In [ ]:
import numpy as np

X = np.array([[0.5, 1.5], [1,1], [1.5, 0.5], [3, 0.5], [2, 2], [1, 2.5]])
y = np.array([0, 0, 0, 1, 1, 1])

与之前一样，使用辅助函数绘制数据：label $y=1$ 的样本用红色叉号表示，label $y=0$ 的样本用黑色圆圈表示。


In [ ]:
from lab_utils import plot_data
import matplotlib.pyplot as plt

plot_data(X,y)

# Set both axes to be from 0-6
plt.axis([0, 6, 0, 6])
# Set the y-axis label
plt.ylabel('$x_2$')
# Set the x-axis label
plt.xlabel('$x_1$')

## Gradient Descent

先实现非 vectorized 版本的 gradient，再实现 vectorized 版本。

### 非 Vectorized 版本

Gradient descent 的每次迭代都会同时更新全部 $w_j$：

$$
w_j:=w_j-\alpha\frac{\partial J(\mathbf{w})}{\partial w_j}
$$

其中：

$$
\frac{\partial J(\mathbf{w})}{\partial w_j}=\frac{1}{m}\sum_{i=1}^{m}\left(f_{\mathbf{w}}(\mathbf{x}^{(i)})-y^{(i)}\right)x_j^{(i)}
$$

**注意：** 其形式与 linear regression 的 gradient 相同，但两种模型对 $f_{\mathbf{w}}(\mathbf{x})$ 的定义不同，因此实际计算也不同。

本实验将实现 $\frac{\partial J(\mathbf{w})}{\partial w_j}$。这里 $m$ 为训练样本数，$f_{\mathbf{w}}(\mathbf{x}^{(i)})$ 为预测值，$y^{(i)}$ 为实际 label。

对于本数据集，模型为：

$$
f_{\mathbf{w}}(\mathbf{x})=g(w_0+w_1x_1+w_2x_2),\qquad g(z)=\frac{1}{1+e^{-z}}
$$

**预处理：** 为便于实现，在 `X` 中增加一列全为 1 的 $x_0$，使线性部分写成 $w_0x_0+w_1x_1+w_2x_2$。这样可以通过循环累加 $w_jx_j$ 来计算预测值。


In [ ]:
# Add a column to X_orig to account for the w_0 term
X_mod = np.hstack([np.ones((X.shape[0],1)), X])

print(X_mod)

### 补充说明：Sigmoid 的实现

`sigmoid` 函数已经提供，可以直接导入并使用，如下所示。


In [ ]:
from lab_utils import sigmoid 

print(sigmoid(0))

### 实现

下面实现非 vectorized 版本。已有代码框架会：

1. 创建与 `w` 的 shape 相同、初始值全为零的 `dw`，用于存储并返回 gradient。
2. 在外层 loop 中依次计算 `dw[j]`。
3. 在内层 loop 中遍历全部样本，计算每个样本对该 gradient 分量的贡献，并存入 list。
4. 对这些贡献求和、除以样本数，得到 `dw[j]`。

参数 `w` 以 list 或 array 的形式传入。

**练习：** 在内层 loop 中先计算 index 为 $i$ 的样本预测值。为避免覆盖外层 loop 的 `j`，这里使用 `k` 遍历 feature：

```python
z = 0
for k in range(n):
    z += w[k] * X[i][k]
f = sigmoid(z)
```

也可以使用 dot product：

```
z = np.dot(w.T, X[i])
f = sigmoid(z)
```

然后计算该样本对第 `j` 个 gradient 分量的贡献：

```
gradient = (f-y[i])*X[i][j]
```


In [ ]:
def compute_gradient(X, y, w):
    # Here X is assumed to pre-processed with a column of ones added as x_0
    m, n = X.shape
    dw = np.zeros_like(w)
    
    for j in range(n):
        gradient_list = []
        
        for i in range(m):        
            ### START CODE HERE ### 
             
            ### END CODE HERE ### 
            gradient_list.append(gradient)
        
        dw[j] = (1/m)* sum(gradient_list)
        
    return dw

In [ ]:
Check the implementation of your gradient function using the cell below.

In [ ]:
w = np.zeros(3)
compute_gradient(X_mod,y,w)

**预期输出：**

```text
array([ 0.        , -0.25      , -0.16666667])
```


### 可选：Vectorized 版本

下面实现 gradient 的 vectorized 版本：

$$
\nabla_{\mathbf{w}}J(\mathbf{w})=\frac{1}{m}\mathbf{X}^T(\mathbf{f}-\mathbf{y}),\qquad \mathbf{f}=g(\mathbf{X}\mathbf{w})
$$

其中，$g$ 是逐元素作用的 sigmoid function。

**练习：** 先计算全部样本的预测值：

```
z = np.dot(X, w)
f = sigmoid(z)
```

然后计算 gradient：

```
dw = (1/m)*np.dot(X.T, (f - y))
```

**调试提示：** Vectorization 出错时，可以打印 array 的 shape，检查维度是否匹配。例如，$\mathbf{X}$ 为 $6\times3$ 的 matrix、$\mathbf{w}$ 为 $3\times1$ 的 vector 时，$\mathbf{X}\mathbf{w}$ 是合法运算，而 $\mathbf{w}\mathbf{X}$ 不是。


In [ ]:
def compute_gradient_vectorized(X, y, w):
    
    m, n = X.shape
    
    ### START CODE HERE ### 
    
    ### END CODE HERE ### 
    return dw

下面检查该函数的输出是否与前面的非 vectorized 实现一致。


In [ ]:
print("Cost computed by non-vectorized version: ", compute_gradient(X_mod, y, w))
print("Cost computed by vectorized version: ", compute_gradient_vectorized(X_mod, y, w))

**预期输出：**

非 vectorized 与 vectorized 版本计算得到的 gradient 都应为：

```text
[ 0.         -0.25       -0.16666667]
```
